In [ ]:
import cv2
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk
import pytesseract
import re
import os
import datetime

# Set Tesseract path (update this path if you're on Windows)
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

current_image = None
current_image_source = None

def show_confirm_buttons():
    confirm_btn.pack(pady=5)
    retake_btn.pack(pady=5)

def hide_confirm_buttons():
    confirm_btn.pack_forget()
    retake_btn.pack_forget()

def display_image_pil(pil_img):
    global current_image
    current_image = pil_img
    img_resized = pil_img.resize((300, 200))
    img_tk = ImageTk.PhotoImage(img_resized)
    panel.config(image=img_tk)
    panel.image = img_tk
    show_confirm_buttons()

def upload_image():
    global current_image_source
    file_path = filedialog.askopenfilename(filetypes=[("Image files", "*.jpg *.jpeg *.png")])
    if file_path:
        pil_img = Image.open(file_path)
        current_image_source = 'upload'
        display_image_pil(pil_img)

def take_picture():
    global current_image_source
    cap = cv2.VideoCapture(0)

    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    rect_w, rect_h = 400, 280
    rect_x = (frame_width - rect_w) // 2
    rect_y = (frame_height - rect_h) // 2

    captured_frame = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        cv2.rectangle(frame, (rect_x, rect_y), (rect_x + rect_w, rect_y + rect_h), (0, 255, 0), 2)
        cv2.putText(frame, "Press 's' to capture, 'q' to quit", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.imshow("Capture ID Card", frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('s'):
            captured_frame = frame[rect_y:rect_y + rect_h, rect_x:rect_x + rect_w]
            break
        elif key == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    if captured_frame is not None:
        pil_img = Image.fromarray(cv2.cvtColor(captured_frame, cv2.COLOR_BGR2RGB))
        current_image_source = 'capture'
        display_image_pil(pil_img)

def extract_name_and_id(image_pil):
    import numpy as np

    # Convert PIL to OpenCV
    img_cv = cv2.cvtColor(np.array(image_pil), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)

    # OCR
    text = pytesseract.image_to_string(thresh)
    print("OCR Result:\n", text)

    lines = [line.strip() for line in text.split("\n") if line.strip()]
    name = ''
    student_id = ''

    for i, line in enumerate(lines):
        # Find student ID using pattern like 24WMR09307
        match = re.search(r'\b\d{2}[A-Z]{3}\d{5}\b', line)
        if match:
            student_id = match.group()

            # Search for name in 2 lines before the ID line
            for j in range(max(0, i - 2), i):
                candidate = lines[j]
                if (
                    len(candidate.split()) >= 2 and
                    candidate.isupper() and
                    not any(k in candidate for k in ['DATE', 'EXPIRY', 'STUDENT', 'TARUMT'])
                ):
                    name = candidate
                    break

            break  # Stop after first match

    return name.strip(), student_id.strip()


def confirm_image():
    global current_image

    if current_image is None:
        messagebox.showwarning("No Image", "Please select or capture an image first.")
        return

    name, student_id = extract_name_and_id(current_image)

    if not name or not student_id:
        messagebox.showwarning("OCR Failed", "Could not extract name or student ID.")
        return

    confirm = messagebox.askyesno("Confirm Student Info", f"Name: {name}\nStudent ID: {student_id}\n\nIs this correct?")

    if confirm:
        folder_name = "StudentidFolder"
        os.makedirs(folder_name, exist_ok=True)

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"{student_id}_{timestamp}.jpg"
        save_path = os.path.join(folder_name, filename)

        current_image.save(save_path)
        messagebox.showinfo("Image Saved", f"Image saved to:\n{save_path}")
        hide_confirm_buttons()
    else:
        messagebox.showinfo("Try Again", "Please retake or reselect the image.")

def retake_or_reselect():
    hide_confirm_buttons()
    panel.config(image=None)
    panel.image = None
    global current_image
    current_image = None

# GUI setup
root = tk.Tk()
root.title("ID Card Image Capture")

upload_btn = tk.Button(root, text="Upload Image", command=upload_image)
upload_btn.pack(pady=10)

capture_btn = tk.Button(root, text="Take a Picture", command=take_picture)
capture_btn.pack(pady=10)

panel = tk.Label(root)
panel.pack(padx=10, pady=10)

confirm_btn = tk.Button(root, text="✅ Confirm", command=confirm_image)
retake_btn = tk.Button(root, text="🔁 Retake/Reselect", command=retake_or_reselect)

root.mainloop()


OCR Result:
 irr ed

WONG WAI FENG
24WMR09307

JSONHY DATE: L-e-1eae,

} mR


